# story_generation_v2 — Playback Manifest E2E

End-to-end validation of the `story_generation_v2` workflow against a live Docker deployment.

**What this notebook covers:**
1. Firebase authentication
2. Service health check
3. Workflow registration check
4. Task creation and polling
5. Manifest shape validation
6. Asset URL reachability
7. Inline preview of generated images and audio

## 0. Configuration

Edit the variables below before running. All other cells are hands-off.

In [ ]:
# ── Target service ────────────────────────────────────────────────────────────
BASE_URL = "http://34.44.73.189:8002"   # Tasks API base URL

# ── Firebase ID token ─────────────────────────────────────────────────────────
FIREBASE_ID_TOKEN = "<secret>"   # paste token here

# ── Story parameters ───────────────────────────────────────────────────────────
STORY_TOPIC       = "sharing toys kindly before bedtime"
STORY_AGE         = 5
STORY_LANGUAGE    = "en"
STORY_TYPE        = "daily_life_bedtime"
STORY_CHARACTER   = "puppy"
STORY_STYLE       = "coco_style"
GENDER_PREFERENCE = "neutral"

# ── Polling / timeout ─────────────────────────────────────────────────────────
TIMEOUT_SECONDS = 900
POLL_INTERVAL   = 3.0

# ── Asset check ───────────────────────────────────────────────────────────────
CHECK_ASSETS = True   # HEAD-probe every imageUrl / audioUrl in the manifest

## 1. Helpers

In [9]:
from __future__ import annotations

import base64
import json
import time
import urllib.error
import urllib.parse
import urllib.request

FIREBASE_SIGN_IN_URL = (
    "https://identitytoolkit.googleapis.com/v1/accounts:signInWithPassword"
)


def http_json(
    method: str,
    url: str,
    *,
    body: dict | None = None,
    headers: dict | None = None,
    timeout: int = 30,
    expect_status: int | None = None,
) -> tuple[int, dict]:
    request_headers = {"Content-Type": "application/json"}
    if headers:
        request_headers.update(headers)
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, headers=request_headers, method=method)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            status = resp.status
            parsed = json.loads(resp.read().decode()) or {}
    except urllib.error.HTTPError as exc:
        status = exc.code
        parsed = json.loads(exc.read().decode(errors="replace") or "{}") or {}
    except urllib.error.URLError as exc:
        raise RuntimeError(f"{method} {url} failed: {exc}") from exc
    if expect_status is not None and status != expect_status:
        raise RuntimeError(
            f"{method} {url} — expected {expect_status}, got {status}:\n"
            + json.dumps(parsed, indent=2, ensure_ascii=False)
        )
    return status, parsed


def decode_jwt_payload(token: str) -> dict:
    part = token.split(".")[1]
    part += "=" * (-len(part) % 4)
    return json.loads(base64.urlsafe_b64decode(part.encode("ascii")))


def firebase_sign_in(api_key: str, email: str, password: str) -> str:
    url = f"{FIREBASE_SIGN_IN_URL}?{urllib.parse.urlencode({'key': api_key})}"
    _, data = http_json(
        "POST", url,
        body={"email": email, "password": password, "returnSecureToken": True},
        headers={},
        expect_status=200,
    )
    token = data.get("idToken")
    if not token:
        raise RuntimeError(f"Firebase sign-in returned no idToken: {data}")
    return token


def auth_headers(token: str) -> dict:
    return {"Authorization": f"Bearer {token}"}


def poll_until_done(
    base_url: str,
    task_id: str,
    headers: dict,
    *,
    timeout_seconds: int,
    poll_interval: float,
) -> dict:
    deadline = time.time() + timeout_seconds
    url = f"{base_url}/api/v1/tasks/{task_id}"
    while time.time() < deadline:
        _, payload = http_json("GET", url, headers=headers)
        status = payload.get("status")
        stage  = payload.get("current_stage")
        prog   = payload.get("stage_progress") or {}
        print(f"  status={status}  stage={stage}  progress={json.dumps(prog)}")
        if status in {"completed", "failed"}:
            return payload
        time.sleep(poll_interval)
    raise TimeoutError(f"Task {task_id} did not complete within {timeout_seconds}s")


print("Helpers loaded.")

Helpers loaded.


## 2. Authentication

In [10]:
if not FIREBASE_ID_TOKEN:
    raise RuntimeError("FIREBASE_ID_TOKEN is empty — paste a valid token in the config cell above.")

jwt = decode_jwt_payload(FIREBASE_ID_TOKEN)
firebase_uid = jwt.get("user_id") or jwt.get("uid") or jwt.get("sub")
if not firebase_uid:
    raise RuntimeError("Token payload has no uid/user_id/sub field.")

HEADERS = auth_headers(FIREBASE_ID_TOKEN)
print(f"Token valid  uid={firebase_uid}")

Token valid  uid=cB1BQHSUjQcWDHoT1FFvTCoLNHk1


## 3. Service Health Check

In [11]:
_, health = http_json("GET", f"{BASE_URL}/health", expect_status=200)
print(json.dumps(health, indent=2))
assert health.get("status") == "ok", f"Unexpected health status: {health}"
print("\n[PASS] Service is healthy.")

{
  "status": "ok"
}

[PASS] Service is healthy.


## 4. Workflow Registration Check

Verifies that `story_generation_v2` is registered and exposes the expected stages.

In [12]:
_, wf_payload = http_json("GET", f"{BASE_URL}/api/v1/tasks/workflows", expect_status=200)

workflows = wf_payload.get("workflows", [])
target = next((w for w in workflows if w.get("name") == "story_generation_v2"), None)
assert target, "story_generation_v2 is not registered"

expected_stages = ["generating_story", "parallel_generation", "manifest_assembly"]
assert target.get("stages") == expected_stages, (
    f"Unexpected stages: {target.get('stages')}"
)

print(f"Registered workflows: {[w['name'] for w in workflows]}")
print(f"\nstory_generation_v2 stages: {target['stages']}")
print("\n[PASS] story_generation_v2 is registered with correct stages.")

Registered workflows: ['ping', 'batch_process', 'story_gen_example_1', 'story_gen_example_2', 'story_generation_v1', 'story_generation_v2', 'tts_preset_v1', 'tts_clone_v1']

story_generation_v2 stages: ['generating_story', 'parallel_generation', 'manifest_assembly']

[PASS] story_generation_v2 is registered with correct stages.


## 5. Create Task & Poll to Completion

In [13]:
create_body = {
    "workflow_name": "story_generation_v2",
    "inputs": {
        "user_id":          firebase_uid,
        "age":              STORY_AGE,
        "language":         STORY_LANGUAGE,
        "story_type":       STORY_TYPE,
        "character":        STORY_CHARACTER,
        "style":            STORY_STYLE,
        "gender_preference": GENDER_PREFERENCE,
        "specific_topic":   STORY_TOPIC,
    },
}

_, create_resp = http_json(
    "POST", f"{BASE_URL}/api/v1/tasks",
    body=create_body, headers=HEADERS, expect_status=200,
)
task_id = create_resp["id"]
print(f"Task created  id={task_id}")
print(json.dumps(create_resp, indent=2, default=str))

Task created  id=3e233ff2-8be2-49c2-b39d-a387d01c53a5
{
  "id": "3e233ff2-8be2-49c2-b39d-a387d01c53a5",
  "workflow_name": "story_generation_v2",
  "status": "pending",
  "created_at": "2026-05-05T06:25:27.445829Z",
  "estimated_seconds": 70,
  "poll_interval_ms": 2000
}


In [14]:
print(f"Polling task {task_id} (timeout={TIMEOUT_SECONDS}s, interval={POLL_INTERVAL}s) ...\n")

final = poll_until_done(
    BASE_URL, task_id, HEADERS,
    timeout_seconds=TIMEOUT_SECONDS,
    poll_interval=POLL_INTERVAL,
)

print("\nFinal status:")
print(json.dumps(final, indent=2, ensure_ascii=False))

if final.get("status") != "completed":
    raise AssertionError(f"Task failed: {final.get('error_message')}")

print("\n[PASS] Task completed successfully.")

Polling task 3e233ff2-8be2-49c2-b39d-a387d01c53a5 (timeout=900s, interval=3.0s) ...

  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress={}
  status=running  stage=generating_story  progress=

## 6. Fetch & Validate Outputs

In [15]:
_, outputs = http_json(
    "GET", f"{BASE_URL}/api/v1/tasks/{task_id}/outputs",
    headers=HEADERS, expect_status=200,
)

manifest = outputs.get("outputs", {})
pages     = manifest.get("pages", [])
total     = manifest.get("totalPages", 0)

# Shape assertions
assert isinstance(manifest, dict),          "outputs is not a dict"
assert isinstance(pages, list) and pages,   "pages list is empty"
assert total == len(pages),                 f"totalPages={total} != len(pages)={len(pages)}"

required_fields = ("pageIndex", "text", "imageUrl", "audioUrl", "audioDurationMs")
for page in pages:
    missing = [f for f in required_fields if f not in page]
    assert not missing, f"Page {page.get('pageIndex')} missing fields: {missing}"
    assert page["pageIndex"] == pages.index(page) + 1, "pageIndex out of order"
    assert page["text"].strip(),              f"Page {page['pageIndex']} has empty text"
    assert page["imageUrl"].strip(),          f"Page {page['pageIndex']} missing imageUrl"
    assert page["audioUrl"].strip(),          f"Page {page['pageIndex']} missing audioUrl"
    assert page["audioDurationMs"] > 0,       f"Page {page['pageIndex']} invalid audioDurationMs"

print(f"storyId    : {manifest.get('storyId')}")
print(f"totalPages : {total}")
print("\n[PASS] Manifest shape is valid.")

storyId    : 3e233ff2-8be2-49c2-b39d-a387d01c53a5
totalPages : 5

[PASS] Manifest shape is valid.


## 7. Asset URL Reachability

In [16]:
if CHECK_ASSETS:
    print("Probing asset URLs ...\n")
    for page in pages:
        for field in ("imageUrl", "audioUrl"):
            url = page[field]
            req = urllib.request.Request(url, method="HEAD")
            try:
                with urllib.request.urlopen(req, timeout=20) as resp:
                    status_code = resp.status
            except urllib.error.HTTPError as exc:
                if exc.code == 405:   # HEAD not allowed — try GET
                    req2 = urllib.request.Request(url, method="GET")
                    with urllib.request.urlopen(req2, timeout=20) as resp:
                        status_code = resp.status
                else:
                    raise AssertionError(
                        f"Page {page['pageIndex']} {field} unreachable: HTTP {exc.code}"
                    ) from exc
            assert 200 <= status_code < 400, (
                f"Page {page['pageIndex']} {field} returned {status_code}"
            )
            print(f"  page {page['pageIndex']}  {field:<12}  HTTP {status_code}  OK")
    print("\n[PASS] All asset URLs are reachable.")
else:
    print("CHECK_ASSETS=False — skipped.")

Probing asset URLs ...

  page 1  imageUrl      HTTP 200  OK
  page 1  audioUrl      HTTP 200  OK
  page 2  imageUrl      HTTP 200  OK
  page 2  audioUrl      HTTP 200  OK
  page 3  imageUrl      HTTP 200  OK
  page 3  audioUrl      HTTP 200  OK
  page 4  imageUrl      HTTP 200  OK
  page 4  audioUrl      HTTP 200  OK
  page 5  imageUrl      HTTP 200  OK
  page 5  audioUrl      HTTP 200  OK

[PASS] All asset URLs are reachable.


## 8. Inline Preview

Renders each page's image and audio player directly in the notebook.

In [17]:
from IPython.display import display, Image, Audio, HTML

for page in pages:
    idx      = page["pageIndex"]
    text     = page["text"]
    img_url  = page["imageUrl"]
    audio_url = page["audioUrl"]
    dur_s    = page["audioDurationMs"] / 1000

    display(HTML(f"<hr><h3>Page {idx}</h3><p>{text}</p>"))
    display(Image(url=img_url, width=512))
    display(HTML(
        f'<audio controls style="margin-top:6px">'
        f'<source src="{audio_url}"></audio>'
        f'<span style="margin-left:8px;color:#555">{dur_s:.1f}s</span>'
    ))

## 9. Summary

In [22]:
print("=" * 60)
print("story_generation_v2 playback manifest E2E — PASSED")
print("=" * 60)
print(f"  task_id    : {task_id}")
print(f"  storyId    : {manifest['storyId']}")
print(f"  totalPages : {manifest['totalPages']}")
for page in pages:
    print(
        f"\n  page {page['pageIndex']}  audio={page['audioDurationMs']}ms\n"
        f"  {page['text']}"
    )

story_generation_v2 playback manifest E2E — PASSED
  task_id    : 3e233ff2-8be2-49c2-b39d-a387d01c53a5
  storyId    : 3e233ff2-8be2-49c2-b39d-a387d01c53a5
  totalPages : 5

  page 1  audio=8335ms
  Pip the puppy had the most wonderful collection of toys. Of all their treasures, the bright red ball was Pip’s absolute favorite.

  page 2  audio=13119ms
  One cozy evening, Pip's friend, a fluffy little bunny named Rosie, hopped over for a quick visit. 'Could I maybe play with your red ball, just for a little bit?' she asked, looking hopefully at Pip.

  page 3  audio=13537ms
  Mummy Dog, tidying nearby, noticed Pip's hesitation. She knelt down gently. 'Remember, Pip,' Mummy Dog said with a soft smile, 'sharing our toys makes playtime much happier for everyone.'

  page 4  audio=11192ms
  Taking a deep breath, Pip slowly pushed the red ball towards Rosie. 'We can play with it together!' Rosie’s ears perked right up, and a wide, joyful grin spread across her face.

  page 5  audio=10356ms
 

## 10. Full Manifest JSON

In [24]:
from IPython.display import display, JSON

display(JSON(outputs))

<IPython.core.display.JSON object>